# Create Whitehall Foundation Awards

Creates the grants of the **Whitehall Foundation** (US; basic research in vertebrate and
invertebrate neurobiology) from the foundation's own published grant lists.

**Prerequisite:** `scripts/local/whitehall_to_s3.py` (uploads the parquet).

**Sources** (both the foundation's own publication; whitehall.org has no robots.txt):
1. The live "Active Grants" list (https://whitehall.org/#active-grants, content file
   `pages/active-grants.html`): 48 active grants with the season + year awarded.
2. The foundation's earlier yearly "Grant Recipients" / "Grant Awards" / "Active Grants" lists
   at whitehall.org/recipien.htm (1998-2001) and whitehall.org/recipients/ (2002-2025),
   no longer online, read from every distinct Internet Archive capture (34 captures, list
   years 1998-2008 and 2011-2024; no capture survives for the 2009 and 2010 lists).

Each list entry is one grant: PI (surname, given), institution, quoted project title,
programme ("Three Year Research Grant", "One Year Grant-in-Aid", "Two Year Renewal Grant")
and total amount in USD. The lists show all ACTIVE grants, so a grant appears on up to
3-4 consecutive lists; entries are merged on (surname, first initial, renewal flag,
title, fuzzy >= 0.9). **713 grants**, 100% title / PI / institution / amount.

**Year:** `start_year` = the live page's "awarded Season YYYY" when present, otherwise the
first list year the grant appears on, used only when the previous year's list was also
captured (so the grant was demonstrably new that year). Checked against the live page,
spring/summer awards first appear on that year's list and fall awards on the next year's,
so this year can be one year late for fall awards. 134 grants first listed on the 1998
list (the oldest capture) or the 2011 list (after the 2009-2010 gap) get `start_year`
NULL (raw `first_listed_year` keeps the list year). No start/end dates published.

**Amount:** the list's "totaling $X" (USD, total over the grant term).

**`funder_award_id` (2.1.1):** Whitehall publishes no grant numbers; citing works quote
internal numbers like `2017-12-98` that never appear on the site, so the key is the
synthetic `WHF-{year}-{surname}-{title hash}` (`WHF-c{list year}-...` when the year is
unbounded). 0 existing citation stubs collapse onto these awards.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306553`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 00her7k05, doi 10.13039/100001391)

**Priority:** `578` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.whitehall_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/whitehall/whitehall_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(DISTINCT funder_award_id) as distinct_ids,
       COUNT(award_year) as with_year, MIN(first_listed_year) as min_list, MAX(first_listed_year) as max_list
FROM openalex.awards.whitehall_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.whitehall_raw;

In [ ]:
%sql
SELECT program, renewal, COUNT(*) AS n, ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_amount
FROM openalex.awards.whitehall_raw GROUP BY 1, 2 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320306553 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320306553;

## Step 2: Create Whitehall Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.whitehall_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306553  -- Whitehall Foundation
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    TRIM(g.title) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CASE WHEN g.program RLIKE '(?i)in[- ]aid' THEN 'Grant-in-Aid'
         WHEN LOWER(g.renewal) = 'true' THEN 'Renewal Grant'
         ELSE 'Research Grant' END as funder_scheme,
    'whitehall_grant_lists' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.award_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.whitehall_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)
Synthetic keys only: `WHF-{yyyy}-{surname slug}-{6 hex}` or `WHF-c{yyyy}-...` (unbounded year).

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.whitehall_awards
WHERE NOT funder_award_id RLIKE '^WHF-c?(19|20)[0-9]{2}-[a-z0-9-]+-[0-9a-f]{6}$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.whitehall_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'whitehall_grant_lists' AND priority = 578;

-- Priority 578: direct-from-funder ingest of the foundation's own grant lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    578 as priority
FROM openalex.awards.whitehall_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.whitehall_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.whitehall_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.whitehall_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.whitehall_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.whitehall_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi
FROM openalex.awards.whitehall_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'whitehall_grant_lists'
GROUP BY provenance, priority;